In [ ]:
# @title 1 · Install Node.js 24, REA, and Python packages

!curl -fsSL https://deb.nodesource.com/setup_24.x | bash - > /dev/null 2>&1
!apt-get install -y -qq nodejs > /dev/null 2>&1
!npm install --global rea-agents@latest --loglevel=error
!pip install -q --upgrade openai mcp
!echo "node $(node --version)" && rea --help | head -n 5

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
added 176 packages in 22s
⠏
⠏40 packages are looking for funding
⠏  run `npm fund` for details
⠏node v24.22.1
rea@6.8.0 — Reverse engineer anything from your terminal or agent.

Usage: rea <command>

Commands:


In [ ]:
# @title 2 · OpenAI key (from Colab Secrets) and model

import os
from google.colab import userdata
from openai import OpenAI
from IPython.display import Markdown, display

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
MODEL = "gpt-5"  # @param {type:"string"}
client = OpenAI()
print("Using model:", MODEL)

Using model: gpt-5


In [ ]:
# @title 3 · Pick the project to analyze

SOURCE = "github_url"  # @param ["github_url", "colab_or_drive_path", "upload_file_or_zip"]
REPO_URL = "https://github.com/morluto/rea"  # @param {type:"string"}
BRANCH = ""  # @param {type:"string"}
PRIVATE_REPO = False  # @param {type:"boolean"}
LOCAL_PATH = "/content/drive/MyDrive/my_project"  # @param {type:"string"}

import pathlib, re, shutil, subprocess, zipfile, json, collections, time

WORK = pathlib.Path("/content/rea_work")
WORK.mkdir(exist_ok=True)

def get_target():
    if SOURCE == "github_url":
        m = re.match(r"https?://github\.com/([^/]+)/([^/#?]+)", REPO_URL.strip())
        if not m:
            raise ValueError("Expected a URL like https://github.com/owner/repo")
        owner, name = m.group(1), m.group(2).removesuffix(".git")
        dest = WORK / name
        shutil.rmtree(dest, ignore_errors=True)
        auth = f"{userdata.get('GITHUB_TOKEN')}@" if PRIVATE_REPO else ""
        cmd = ["git", "clone", "--depth", "1"] + (["--branch", BRANCH] if BRANCH else [])
        cmd += [f"https://{auth}github.com/{owner}/{name}.git", str(dest)]
        p = subprocess.run(cmd, capture_output=True, text=True)
        if p.returncode != 0:
            err = p.stderr.replace(auth, "***@") if auth else p.stderr
            raise RuntimeError("git clone failed:\n" + err)
        return dest

    if SOURCE == "colab_or_drive_path":
        if LOCAL_PATH.startswith("/content/drive"):
            from google.colab import drive
            drive.mount("/content/drive")
        path = pathlib.Path(LOCAL_PATH).resolve()
        if not path.exists():
            raise FileNotFoundError(path)
        return path

    from google.colab import files
    uploaded = files.upload()
    name = next(iter(uploaded))
    src = pathlib.Path("/content") / name
    dest = WORK / src.stem
    shutil.rmtree(dest, ignore_errors=True)
    dest.mkdir(parents=True)
    if src.suffix.lower() == ".zip" and zipfile.is_zipfile(src):
        with zipfile.ZipFile(src) as z:
            z.extractall(dest)
    else:
        shutil.move(str(src), dest / name)  # single artifact: .apk, .exe, .asar, ...
    return dest

TARGET = get_target().resolve()
print("Target:", TARGET)

Target: /content/rea_work/rea


In [ ]:
# @title 4 · Survey the project and find artifacts REA can inspect

SKIP = {".git", "node_modules", ".venv", "venv", "env", "__pycache__", ".idea", ".vscode", ".next", ".tox"}
LANG = {".py": "Python", ".js": "JavaScript", ".mjs": "JavaScript", ".cjs": "JavaScript", ".jsx": "JavaScript",
        ".ts": "TypeScript", ".tsx": "TypeScript", ".java": "Java", ".kt": "Kotlin", ".go": "Go", ".rs": "Rust",
        ".c": "C", ".h": "C/C++", ".cpp": "C++", ".cc": "C++", ".cs": "C#", ".rb": "Ruby", ".php": "PHP",
        ".swift": "Swift", ".m": "Objective-C", ".dart": "Dart", ".lua": "Lua", ".sh": "Shell",
        ".ipynb": "Notebook", ".html": "HTML", ".css": "CSS", ".sql": "SQL"}
MANIFESTS = {"package.json", "pyproject.toml", "requirements.txt", "setup.py", "setup.cfg", "Cargo.toml", "go.mod",
             "pom.xml", "build.gradle", "build.gradle.kts", "composer.json", "Gemfile", "Dockerfile",
             "docker-compose.yml", "Makefile", "CMakeLists.txt", "pubspec.yaml"}
ARTIFACT_EXT = {".asar", ".apk", ".exe", ".dll", ".so", ".dylib", ".wasm", ".gguf", ".safetensors", ".pcap",
                ".pcapng", ".har", ".saz", ".ips", ".beam", ".ko", ".pck", ".sqlite", ".db", ".elf", ".bin"}
MAGIC = {b"\x7fELF": "ELF", b"MZ": "PE", b"\xcf\xfa\xed\xfe": "Mach-O", b"\xca\xfe\xba\xbe": "Mach-O/Java",
         b"\x00asm": "WASM", b"GGUF": "GGUF", b"SQLite f": "SQLite"}

def clip(x, n):
    s = x if isinstance(x, str) else json.dumps(x, ensure_ascii=False, default=str)
    return s if len(s) <= n else s[:n] + f"\n…[truncated {len(s) - n} chars]"

def sniff(path):
    try:
        with open(path, "rb") as f:
            head = f.read(8)
    except OSError:
        return None
    return next((kind for magic, kind in MAGIC.items() if head.startswith(magic)), None)

def survey(root, max_tree_lines=300):
    root = pathlib.Path(root)
    tree, langs, manifests, artifacts, js_apps, readme = [], collections.Counter(), {}, [], [], ""
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = sorted(d for d in dirnames if d not in SKIP)
        rel = pathlib.Path(dirpath).relative_to(root)
        depth = len(rel.parts)
        if depth <= 3 and len(tree) < max_tree_lines:
            tree.append("  " * depth + (rel.name or root.name) + "/")
        if "package.json" in filenames:
            js_apps.append(dirpath)
        for fn in sorted(filenames):
            p = pathlib.Path(dirpath) / fn
            if p.is_symlink():
                continue
            ext = p.suffix.lower()
            if ext in LANG:
                langs[LANG[ext]] += 1
            if depth <= 2 and len(tree) < max_tree_lines:
                tree.append("  " * (depth + 1) + fn)
            if fn in MANIFESTS and len(manifests) < 15:
                manifests[str(p.relative_to(root))] = clip(p.read_text(errors="ignore"), 3000)
            if depth == 0 and not readme and fn.lower().startswith("readme"):
                readme = clip(p.read_text(errors="ignore"), 8000)
            magic = sniff(p) if (ext in ARTIFACT_EXT or not ext) else None
            if ext in ARTIFACT_EXT or magic:
                artifacts.append({"path": str(p), "kind": magic or ext, "size": p.stat().st_size})
    return {"tree": tree, "languages": langs, "manifests": manifests, "readme": readme,
            "artifacts": artifacts, "js_apps": js_apps}

SURVEY = survey(TARGET)
print("Languages:", dict(SURVEY["languages"].most_common(10)))
print("Manifests:", list(SURVEY["manifests"]))
print("JS/Electron app folders:", SURVEY["js_apps"][:10])
print("Binary/artifact files:", [(a["kind"], a["path"]) for a in SURVEY["artifacts"][:20]])

Languages: {'TypeScript': 4779, 'JavaScript': 529, 'Python': 108, 'C': 28, 'Java': 24, 'HTML': 23, 'Swift': 20, 'C++': 7, 'Shell': 5, 'Objective-C': 3}
Manifests: ['package.json', 'scripts/verify-tools/skills/package.json', 'tests/conformance/psp/Makefile', 'tests/conformance/readiness/electron/package.json', 'tests/conformance/readiness/javascript-cli/package.json', 'website/requirements.txt', 'website/public/examples/notes-electron/package.json']
JS/Electron app folders: ['/content/rea_work/rea', '/content/rea_work/rea/scripts/verify-tools/skills', '/content/rea_work/rea/tests/conformance/readiness/electron', '/content/rea_work/rea/tests/conformance/readiness/javascript-cli', '/content/rea_work/rea/website/public/examples/notes-electron']
Binary/artifact files: []


In [ ]:
# @title 5 · Run REA's static analysis on what it found

MAX_JS_APPS = 3  # @param {type:"integer"}
MAX_ARTIFACTS = 10  # @param {type:"integer"}

def rea_cli(*args, timeout=900):
    try:
        p = subprocess.run(["rea", *args, "--json"], capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        return {"exit": "timeout"}
    try:
        out = json.loads(p.stdout)
    except json.JSONDecodeError:
        out = p.stdout[-4000:]
    return {"exit": p.returncode, "result": out, "stderr": p.stderr[-1500:]}

rea_results = {"providers": rea_cli("providers"), "javascript": {}, "artifacts": {}}
for app in SURVEY["js_apps"][:MAX_JS_APPS]:
    print("JS/Electron analysis:", app)
    rea_results["javascript"][app] = rea_cli("analyze-javascript-application", app)
for art in SURVEY["artifacts"][:MAX_ARTIFACTS]:
    print("Artifact inspection:", art["path"])
    rea_results["artifacts"][art["path"]] = rea_cli("inspect-artifact", art["path"])

(WORK / "rea_results.json").write_text(json.dumps(rea_results, indent=2, default=str))
for kind in ("javascript", "artifacts"):
    for path, r in rea_results[kind].items():
        print(f"[{kind}] exit={r['exit']}  {path}")

JS/Electron analysis: /content/rea_work/rea
JS/Electron analysis: /content/rea_work/rea/scripts/verify-tools/skills
JS/Electron analysis: /content/rea_work/rea/tests/conformance/readiness/electron
[javascript] exit=1  /content/rea_work/rea
[javascript] exit=0  /content/rea_work/rea/scripts/verify-tools/skills
[javascript] exit=0  /content/rea_work/rea/tests/conformance/readiness/electron


In [ ]:
# @title 6 · Ask GPT: what is this project about?

REPORT_PROMPT = """You are a senior software engineer and reverse engineer.
From the survey and REA evidence below, explain the project in Markdown with these sections:
1. What it is and what problem it solves (one paragraph, plain language)
2. Who it's for / typical use
3. Architecture and main components (cite file paths)
4. Entry points and how data/control flows
5. Tech stack and key dependencies
6. Notable findings from REA's evidence (modules, IPC, binaries, etc.), if any
7. How to build/run it
8. Unknowns and things worth investigating next
Separate observed facts from inferences. Don't invent files or features that aren't in the evidence."""

tree_text = "\n".join(SURVEY["tree"])
context = f"""# Target
{TARGET}

# Language file counts
{dict(SURVEY['languages'].most_common(15))}

# Directory tree (depth <= 3)
{tree_text}

# README
{SURVEY['readme'] or '(none)'}

# Manifests
{clip(SURVEY['manifests'], 20000)}

# REA evidence
{clip(rea_results, 60000)}
"""

resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "system", "content": REPORT_PROMPT}, {"role": "user", "content": context}],
)
report = resp.choices[0].message.content
(WORK / "report.md").write_text(report)
display(Markdown(report))

1) What it is and what problem it solves
REA (Reverse Engineer Anything) is a CLI and MCP server that lets you and your agent inspect binaries, applications, and runtime behavior in one place, returning evidence-backed findings you can trust. It connects to reverse‑engineering tools (Ghidra, Hopper, IDA, JADX, etc.), performs static JavaScript/Electron and managed code analysis, and captures artifacts like UI traces, network traffic, and SQLite schema, so you can understand how a feature works and reimplement it safely.

2) Who it’s for / typical use
- Product engineers and researchers: understand how a feature in another app works, then build it with confidence.
- Reverse engineers and security auditors: inventory code paths, cross-language/native boundaries, IPC, and binary structure with defensible evidence.
- Agents (via MCP): ask multi-step questions about a target, get structured findings with limitations and next steps.

Typical uses include:
- Inspecting Electron/JavaScript apps and boundaries (IPC, contextBridge calls) without running them.
- Headless native analysis of binaries with Ghidra/Hopper/IDA; mapping procedures, strings, segments, and xrefs.
- Inspecting managed (.NET/CLI) assemblies’ metadata and native boundaries.
- Capturing web/network/SQLite artifacts to correlate behavior with code paths.

3) Architecture and main components (observed)
Observed facts
- CLI and MCP server
  - CLI entry points: scripts/rea.mjs (exposed as bins “rea” and “rea-agents” in package.json).
  - MCP server identity: mcpName io.github.morluto/rea (package.json).
  - Node engines: ^22.23.4 || ^24.22.1 || >=26.12.1 (package.json).
  - Server metadata at runtime: rea-agents 6.8.0, CLI commands: 175, MCP tools: 229 (REA evidence: server_identity).
  - Worker processes for analysis: imports map to dist/composition/javascriptAnalysisWorkerProcess.js and dist/composition/webBundleAnalysisWorkerProcess.js (package.json "imports").
- Providers/adapters and bridges
  - Ghidra Java bridge and extensions: bridge/ghidra/*.java (ReaGhidraBridge.java, ReaGhidraApplySeeds.java, …).
  - Hopper bridge: bridge/hopper_bridge.py and a demo script scripts/hopper-demo-x11.py.
  - Cutter bridge: bridge/cutter/rea_bridge.py.
  - IDA adapter (via upstream MCP): configured by REA_IDA_MCP_CONFIG (docs/ida-provider.md; evidenced in provider health).
  - Android/JADX bridge: bridge/android/ReaJadxBridge.java.
  - MITM/network capture: bridge/mitmproxy/capture.py.
  - pwndbg integration: bridge/pwndbg/*.py.
  - pwntools helpers: bridge/pwntools/*.py.
  - Native/UI/process helpers: bridge/native/*.swift, bridge/native/rea_lldb_tracer.py, bridge/process/*.swift.
  - Windows native addon shipped in package: native/windows/build/rea-windows-x64.node (package.json files).
- Built-in analysis providers (non-external engines)
  - Artifact graph inspector: rea-artifact-graph (available).
  - .NET/CLI static inventory: rea-dotnet-static (available).
  - SQLite inspector: sqlite (available; requires modern Node API per provider note).
- Source and tests (detected by JS analyzer)
  - src/… (e.g., src/domain/referenceSourceGraph.test.ts; src/application/AnalysisContextQueries.ts).
  - tests/… (e.g., tests/conformance/…, tests/boundary/filesystem/… referenced in analyzer output).

Inferences
- TypeScript sources compile to dist/; scripts/prepare.mjs and scripts/prepack.mjs likely coordinate builds/publish.
- The CLI routes commands to internal provider orchestrations and spawns workers (per package.json imports) for heavier analyses.

4) Entry points and how data/control flows
Observed facts
- CLI
  - Entry: scripts/rea.mjs (package.json "bin": rea, rea-agents).
  - Common flows (from README and docs): “npx rea-agents setup” registers the MCP server with supported agents; “npx -y rea-agents analyze-javascript-application /path --json” runs static JS/Electron analysis.
  - For native analysis, external tools are discovered/configured via environment (e.g., GHIDRA_INSTALL_DIR; Hopper path; REA_IDA_MCP_CONFIG).
- MCP
  - Server name/version reported at startup; exposes a large tool catalog (REA evidence).
- Providers
  - Built-in providers (artifact graph, .NET/CLI, SQLite) run locally within the Node process and worker subprocesses (imports -> dist/composition/*WorkerProcess.js).
  - External providers: REA launches headless Ghidra, Hopper, IDA, JADX, etc., via their bridges and collects normalized findings (capability sets in provider matrix).

Inferences
- Control flow for static JS analysis: CLI -> spawn JS analysis worker -> parse AST -> retain static facts -> (optionally) expand relationships into an application graph; returns graph + evidence.
- Native control flow: CLI/MCP -> pick configured native provider -> bridge adapter starts the tool headless -> queries (list_procedures, decompile, xrefs, etc.) -> normalize -> return dossier + evidence.

5) Tech stack and key dependencies
Observed facts
- Languages: TypeScript (primary), JavaScript, Python, Swift, Java, C/C++ (fixtures), Shell (language counts).
- Node.js ESM project with vitest (vitest.config.ts), Stryker mutation testing (stryker*.mjs), turbo.json.
- Docs site: VitePress (docs/.vitepress/config.ts), additional website tooling (website/requirements.txt for CairoSVG/polib).
- External tools integrated via bridges: Ghidra, Hopper, IDA MCP, JADX, mitmproxy, pwndbg, pwntools.
- SQLite inspector requires Node 24.x ≥24.15 or 26+ with DatabaseSync.limits (REA provider note).
- Windows native addon: native/windows/build/rea-windows-x64.node.

Inferences
- The project is published on npm as rea-agents and meant to be installed via npm/npx or bunx.
- Bridges imply host-side dependencies (e.g., Hopper installation, Ghidra 12.1.x) that REA does not bundle.

6) Notable findings from REA’s evidence
Observed facts
- Live server identity
  - Package/server: rea-agents 6.8.0; MCP SDK server/client/core 2.3.1.
  - Tool/catalog counts: 175 CLI commands; 229 MCP tools.
- Provider availability on the analysis host
  - rea-artifact-graph: available (extract/inspect).
  - rea-dotnet-static: available (inspect_managed_artifact/members/native_boundaries).
  - sqlite: available (inspect_sqlite_database).
  - ghidra: unavailable (not_configured; set GHIDRA_INSTALL_DIR to 12.1.x; rich capability list shown).
  - hopper: unavailable (launcher missing at /opt/hopper/bin/Hopper).
  - ida: unavailable (not_configured; set REA_IDA_MCP_CONFIG).
  - kuna: unavailable (not_configured; set REA_KUNA_COMMAND).
- Static JavaScript application analysis on this repository
  - Partially executed then stopped due to parent result budget (memory) exhaustion after covering tests/boundary/filesystem/ghidraProfileArtifactOwnership.test.ts (resource_constraint).
  - Complete static facts retained for src/domain/referenceSourceGraph.test.ts (observed imports: node:crypto, canonicalize, vitest, ./referenceSourceGraph.js; endpoints recorded).
  - Semantic analysis of src/application/AnalysisContextQueries.ts was not started due to reserved remaining budget (explicit limitation message).
  - Reported parent result budget ~924 MB; worker heap limit ~1.02 GB; detailed remediation suggests increasing Node’s old-space size via NODE_OPTIONS.

Inferences
- The project’s JS codebase is large enough that full-graph expansion benefits from raising the Node parent heap or focusing on subtrees.

7) How to build/run it
Observed facts
- Quick start (CLI)
  - One-off: run “npx rea-agents setup” to register the MCP server with your agent.
  - Analyze a JavaScript/Electron app directory: “npx -y rea-agents@latest analyze-javascript-application /absolute/path --json”.
  - Install globally: “npm install --global rea-agents”; then use “rea --help” and “rea update” to upgrade.
- Native providers
  - Ghidra: set GHIDRA_INSTALL_DIR to an extracted Ghidra 12.1.x directory (provider diagnostics).
  - Hopper: provide an executable launcher path (provider shows /opt/hopper/bin/Hopper was missing on host).
  - IDA: set REA_IDA_MCP_CONFIG to an upstream IDA MCP JSON registration (docs/ida-provider.md).
  - Kuna: set REA_KUNA_COMMAND to absolute path of kuna executable (verified with 1.833).
- Platform note (README)
  - On Linux, ensure a procps-compatible ps is on PATH for process ownership checks (docs/installation.md).
- SQLite provider
  - Requires Node 24.x ≥24.15 or 26+ (provider note).
- Documentation
  - CLI and evidence guide: docs/cli.md.
  - Provider references and analysis topics: docs/* (e.g., docs/ghidra-provider.md, docs/javascript-application-graph.md, docs/javascript-recovery.md, docs/reverse-engineering-tools.md).

Inferences
- For local development, installing dependencies and running the project likely triggers build steps via scripts/prepare.mjs/prepack.mjs; production packages ship prebuilt dist. Prefer npm/npx usage unless contributing to core.

8) Unknowns and things worth investigating next
Observed unknowns
- Exact internal module layout of dist/ and how commands map to TypeScript sources is not listed in the tree (build outputs not included in repo checkout).
- The purpose and API surface of the Windows native addon native/windows/build/rea-windows-x64.node are not documented in the evidence.
- Many external providers were not configured on the analysis host, so their end-to-end flows weren’t exercised here.

Inferences / next steps
- Configure one native provider (e.g., Ghidra via GHIDRA_INSTALL_DIR) and re-run CLI native commands to validate full function dossiers, decompilation, and xrefs against a sample binary.
- Re-run JavaScript analysis with higher Node old-space (e.g., set NODE_OPTIONS="--max-old-space-size=4096") or analyze a subset path to see the complete application graph across src/.
- Exercise SQLite, .NET static, and artifact-graph providers on sample artifacts to validate their outputs and limitations in your environment.
- Inspect bridges:
  - bridge/mitmproxy/capture.py to understand capture schema and integration commands.
  - bridge/pwndbg/* and bridge/pwntools/* for crash decoding and layout helpers.
  - bridge/android/ReaJadxBridge.java for how Android artifacts are parsed/inventoried.
- Review GitHub workflows (.github/workflows/*) naming “real-*” to see how CI stages drive live tool checks; adapt those steps locally when reproducing analyses.

Appendix: key file paths cited
- CLI/MCP: scripts/rea.mjs, package.json (mcpName io.github.morluto/rea), server.json.
- Analysis workers: dist/composition/javascriptAnalysisWorkerProcess.js, dist/composition/webBundleAnalysisWorkerProcess.js (mapped via package.json "imports").
- Native bridges: bridge/ghidra/*.java, bridge/hopper_bridge.py, bridge/cutter/rea_bridge.py, bridge/android/ReaJadxBridge.java.
- Capture/RE tooling: bridge/mitmproxy/capture.py, bridge/pwndbg/*.py, bridge/pwntools/*.py, bridge/native/*.swift, bridge/native/rea_lldb_tracer.py, bridge/process/*.swift.
- Docs and guides: docs/*.md (e.g., docs/cli.md, docs/ghidra-provider.md, docs/javascript-application-graph.md).
- Website assets/config: docs/.vitepress/config.ts, website/*.
- Tests/fixtures: tests/conformance/*, tests/boundary/* (referenced by analyzer).
- Native Windows addon: native/windows/build/rea-windows-x64.node.

In [ ]:
# @title 7 · Start the REA MCP bridge + GPT agent

import asyncio, threading
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

ALLOW_RUNTIME_TOOLS = False  # @param {type:"boolean"}
RUNTIME_WORDS = {"capture", "launch", "scenario", "adb", "device", "browser", "observe", "observation", "process",
                 "run", "execute", "spawn", "attach", "inspector", "install", "setup", "upgrade", "uninstall"}

def is_runtime(name):
    return bool(set(name.lower().split("_")) & RUNTIME_WORDS)

class ReaMCP:
    def __init__(self, startup_timeout=180):
        self.loop = asyncio.new_event_loop()
        threading.Thread(target=self.loop.run_forever, daemon=True).start()
        self.ready = threading.Event()
        self.log = open(WORK / "rea_mcp.log", "w")
        self.main = asyncio.run_coroutine_threadsafe(self._main(), self.loop)
        t0 = time.time()
        while not self.ready.wait(1):
            if self.main.done():
                self.main.result()  # re-raises the startup error
            if time.time() - t0 > startup_timeout:
                raise RuntimeError(f"REA MCP did not start; see {WORK / 'rea_mcp.log'}")

    async def _main(self):
        env = {k: v for k, v in os.environ.items() if k != "OPENAI_API_KEY"}
        params = StdioServerParameters(command="rea", args=["mcp"], env=env)
        async with stdio_client(params, errlog=self.log) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                self.session = session
                self.tools = {t.name: t for t in (await session.list_tools()).tools}
                self._stop = asyncio.Event()
                self.ready.set()
                await self._stop.wait()

    def call(self, name, arguments, timeout=900):
        fut = asyncio.run_coroutine_threadsafe(self.session.call_tool(name, arguments), self.loop)
        return fut.result(timeout)

    def close(self):
        self.loop.call_soon_threadsafe(self._stop.set)

try:
    REA.close()
except NameError:
    pass
REA = ReaMCP()
print(f"REA MCP ready with {len(REA.tools)} tools")

# ---------- tools GPT can call ----------
def _safe(path):
    p = pathlib.Path(path)
    p = (p if p.is_absolute() else TARGET / p).resolve()
    if not (p.is_relative_to(TARGET) or p.is_relative_to(WORK)):
        raise PermissionError("Path is outside the project folder")
    return p

def list_files(path=".", max_depth=2):
    base, lines = _safe(path), []
    for dp, dns, fns in os.walk(base):
        dns[:] = sorted(d for d in dns if d not in SKIP)
        depth = len(pathlib.Path(dp).relative_to(base).parts)
        if depth > max_depth:
            dns[:] = []
            continue
        lines.append("  " * depth + pathlib.Path(dp).name + "/")
        lines += ["  " * (depth + 1) + f for f in sorted(fns)]
        if len(lines) > 600:
            break
    return clip("\n".join(lines), 15000)

def read_file(path, start_line=1, max_lines=400):
    p = _safe(path)
    lines = p.read_text(errors="ignore").splitlines()
    end = min(len(lines), start_line - 1 + max_lines)
    body = "\n".join(f"{i + 1}: {lines[i]}" for i in range(start_line - 1, end))
    return clip(f"{p} (lines {start_line}-{end} of {len(lines)})\n{body}", 30000)

def search_code(pattern, path="."):
    p = subprocess.run(["grep", "-rnIE", "--exclude-dir=.git", "--exclude-dir=node_modules", "-m", "20",
                        "-e", pattern, str(_safe(path))], capture_output=True, text=True, timeout=120)
    return clip(p.stdout or p.stderr or "no matches", 15000)

def rea_find_tools(query, limit=8):
    usable = [t for t in REA.tools.values() if ALLOW_RUNTIME_TOOLS or not is_runtime(t.name)]
    if query.strip().lower() in ("", "*", "all"):
        return clip(sorted(t.name for t in usable), 15000)
    words = [w for w in re.split(r"\W+", query.lower()) if w]
    scored = []
    for t in usable:
        hay = (t.name + " " + (t.description or "")).lower()
        score = sum(hay.count(w) for w in words) + 3 * sum(w in t.name.lower() for w in words)
        if score:
            scored.append((score, t))
    scored.sort(key=lambda x: -x[0])
    return clip([{"name": t.name, "description": clip(t.description or "", 600), "input_schema": t.inputSchema}
                 for _, t in scored[:limit]], 20000)

def rea_call(tool, arguments=None):
    if tool not in REA.tools:
        return f"Unknown tool '{tool}'. Use rea_find_tools first."
    if not ALLOW_RUNTIME_TOOLS and is_runtime(tool):
        return "Blocked: runtime tool (set ALLOW_RUNTIME_TOOLS=True in cell 7 to enable)."
    try:
        res = REA.call(tool, arguments or {})
    except Exception as e:
        return f"ERROR calling {tool}: {type(e).__name__}: {e}"
    texts = [c.text for c in res.content if getattr(c, "type", "") == "text"]
    if not texts and getattr(res, "structuredContent", None):
        texts = [json.dumps(res.structuredContent, ensure_ascii=False)]
    return clip(("ERROR: " if res.isError else "") + ("\n".join(texts) or "(no text content)"), 20000)

def fn_spec(name, desc, props, required):
    return {"type": "function", "function": {"name": name, "description": desc,
            "parameters": {"type": "object", "properties": props, "required": required}}}

AGENT_TOOLS = [
    fn_spec("rea_find_tools", "Search REA's reverse-engineering tool catalog by keywords (e.g. 'javascript modules', "
            "'open binary', 'apk manifest', 'strings', 'dotnet'). Returns names, descriptions and input schemas. "
            "Query 'all' lists every tool name.",
            {"query": {"type": "string"}, "limit": {"type": "integer"}}, ["query"]),
    fn_spec("rea_call", "Call one REA MCP tool with arguments matching its input schema. Use absolute paths.",
            {"tool": {"type": "string"},
             "arguments": {"type": "object", "additionalProperties": True}}, ["tool", "arguments"]),
    fn_spec("list_files", "List files in the project (path relative to project root or absolute).",
            {"path": {"type": "string"}, "max_depth": {"type": "integer"}}, []),
    fn_spec("read_file", "Read a project file with line numbers.",
            {"path": {"type": "string"}, "start_line": {"type": "integer"}, "max_lines": {"type": "integer"}}, ["path"]),
    fn_spec("search_code", "grep -E search across the project source.",
            {"pattern": {"type": "string"}, "path": {"type": "string"}}, ["pattern"]),
]
LOCAL = {"rea_find_tools": rea_find_tools, "rea_call": rea_call, "list_files": list_files,
         "read_file": read_file, "search_code": search_code}

AGENT_SYSTEM = f"""You are a reverse-engineering and code-comprehension agent running in Google Colab (Linux x64).
Project root: {TARGET}
Languages: {dict(SURVEY['languages'].most_common(10))}
JS/Electron app folders: {SURVEY['js_apps'][:10]}
Binary/artifact files: {clip([(a['kind'], a['path']) for a in SURVEY['artifacts']], 3000)}
A static report was already produced:
{clip(report, 6000)}

Tools:
- list_files / read_file / search_code: read source directly. Prefer these for ordinary source code.
- rea_find_tools + rea_call: REA's evidence-producing analysis (JavaScript/Electron graphs, native binaries,
  APKs, .NET, WASM, model files, captures...). Find the right tool first, then call it with absolute paths.
  If you open a binary/analysis session, close it when finished.
Rules: base every claim on evidence you actually retrieved; cite file paths, symbols or addresses;
separate observed facts, inferences and unknowns; if a tool fails, say why and try another route.
Answer in the user's language."""

messages = [{"role": "system", "content": AGENT_SYSTEM}]

def _compact(msgs, keep_last=6, limit=1500):
    tool_idx = [i for i, m in enumerate(msgs) if m.get("role") == "tool"]
    for i in tool_idx[:-keep_last]:
        msgs[i]["content"] = clip(msgs[i]["content"], limit)

def ask(question, max_steps=30, verbose=True):
    messages.append({"role": "user", "content": question})
    for _ in range(max_steps):
        _compact(messages)
        msg = client.chat.completions.create(model=MODEL, messages=messages, tools=AGENT_TOOLS).choices[0].message
        if not msg.tool_calls:
            messages.append({"role": "assistant", "content": msg.content})
            display(Markdown(msg.content or ""))
            return msg.content
        messages.append({"role": "assistant", "content": msg.content,
                         "tool_calls": [tc.model_dump() for tc in msg.tool_calls]})
        for tc in msg.tool_calls:
            try:
                args = json.loads(tc.function.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            fn = LOCAL.get(tc.function.name)
            try:
                out = fn(**args) if fn else f"Unknown tool {tc.function.name}"
            except Exception as e:
                out = f"ERROR: {type(e).__name__}: {e}"
            if verbose:
                print(f"→ {tc.function.name} {clip(json.dumps(args, ensure_ascii=False), 160)}  ({len(out)} chars)")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(out)})
    print("Stopped after max_steps — ask a follow-up to continue.")

def reset_chat():
    del messages[1:]

REA MCP ready with 229 tools


In [ ]:
# @title 8 · Ask the agent
QUESTION = "What is this project about? Walk me through its architecture, main entry points and data flow, with evidence."  # @param {type:"string"}
answer = ask(QUESTION)

# Save the whole Q&A transcript
transcript = "\n\n".join(f"## {m['role']}\n{m['content']}" for m in messages[1:]
                         if m["role"] in ("user", "assistant") and m.get("content"))
(WORK / "transcript.md").write_text(transcript)

→ list_files {"path": "/content/rea_work/rea", "max_depth": 2}  (15024 chars)
→ read_file {"path": "/content/rea_work/rea/package.json", "start_line": 1, "max_lines": 400}  (28489 chars)
→ read_file {"path": "/content/rea_work/rea/scripts/rea.mjs", "start_line": 1, "max_lines": 400}  (11439 chars)
→ list_files {"path": "/content/rea_work/rea/src", "max_depth": 2}  (15024 chars)
→ read_file {"path": "/content/rea_work/rea/src/main.ts", "start_line": 1, "max_lines": 400}  (7822 chars)
→ read_file {"path": "/content/rea_work/rea/src/cli.ts", "start_line": 1, "max_lines": 400}  (10363 chars)
→ read_file {"path": "/content/rea_work/rea/src/identity.ts", "start_line": 1, "max_lines": 200}  (1237 chars)
→ read_file {"path": "/content/rea_work/rea/src/composition/directAnalysis.ts", "start_line": 1, "max_lines": 400}  (4009 chars)
→ list_files {"path": "/content/rea_work/rea/src/cli", "max_depth": 1}  (1726 chars)
→ search_code {"pattern": "javascriptAnalysisWorkerProcess", "path": "/content/r

117